# 1. Designing a base editor library

`be_scan.sgrna` turns a gene sequence into an annotated sgRNA library. This
notebook walks through the four steps individually and then shows the single
call that runs all of them.

The example gene is **FOXA1** (chr14, minus strand, 2 exons, 473 residues).
The input is a UCSC-style `.fasta` with one record per exon, exons in
uppercase and 30 nt of flanking intron in lowercase.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

DATA = "data"
GENE = f"{DATA}/240401_FOXA1_Input_30.fasta"

print(open(GENE).read()[:320], "...")

>hg38_knownGene_ENST00000250448.5_0 range=chr14:37594871-37595002 5'pad=30 3'pad=30 strand=- repeatMasking=none
gatggttgtattgggcagggtggctccaggATGTTAGGAACTGTGAAGAT
GGAAGGGCATGAAACCAGCGACTGGAACAGCTACTACGCAGACACGCAGG
AGgtgagaggcaaggcgaggcgggcggccggt
>hg38_knownGene_ENST00000250448.5_1 range=chr14:37591335-37592741 5'pad=3 ...


## Step 1 — `generate_library`

Finds every guide with a matching PAM and, optionally, an editable base in
the window.

Two parameters are worth calling out:

- **`window`** is the editing window in 1-indexed guide coordinates. `[4,8]`
  is the default and matches most BE3/BE4/ABE constructs.
- **`guide_len`** is new in 3.0.0. It defaults to 20, and the searched
  sequence is `guide_len + len(PAM)`, so shorter spacers and non-3 nt PAMs
  both work.

In [2]:
from be_scan.sgrna import generate_library

guides, gene = generate_library(
    gene_filepath = GENE,
    gene_name     = "FOXA1",
    cas_type      = "SpG",     # NGN; see cas_key for the rest
    edit_from     = "A",       # ABE
    edit_to       = "G",
    window        = [4, 8],
    return_df     = True,
    save_df       = False,
)
print(guides.shape)
guides.head()

Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated
(992, 13)


,sgRNA_seq,PAM_seq,starting_frame,gene_pos,chr_pos,exon,windowstart_pos,windowend_pos,UTR,coding_seq,sgRNA_strand,gene_strand,gene
0,gatggttgtattgggcaggg,tgg,0,-1,37594871,0,-1,-1,True,gatggttgtattgggcaggg,sense,minus,FOXA1
1,atggttgtattgggcagggt,ggc,1,-1,37594870,0,-1,-1,True,atggttgtattgggcagggt,sense,minus,FOXA1
2,gtattgggcagggtggctcc,agg,1,-1,37594864,0,-1,-1,True,gtattgggcagggtggctcc,sense,minus,FOXA1
3,tattgggcagggtggctcca,ggA,2,-1,37594863,0,-1,-1,True,tattgggcagggtggctcca,sense,minus,FOXA1
4,tgggcagggtggctccaggA,TGT,2,-1,37594860,0,-1,-1,True,tgggcagggtggctccaggA,sense,minus,FOXA1


The gene object carries the parsed structure, which is worth a look before
trusting any annotation built on top of it.

In [3]:
cds = "".join(gene.exons)
print("chromosome     :", gene.chrID)
print("strand         :", gene.strand)
print("exons          :", len(gene.exons), [len(e) for e in gene.exons])
print("intron padding :", gene.intron_len, "nt")
print("CDS length     :", len(cds), "nt ->", len(cds)//3, "codons")
print("starts with ATG:", cds.startswith("ATG"))

chromosome     : chr14
strand         : minus
exons          : 2 [72, 1347]
intron padding : 30 nt
CDS length     : 1419 nt -> 473 codons
starts with ATG: True


### Choosing a Cas variant

`cas_type` selects a PAM; `PAM` overrides it with anything you like, using
IUPAC codes (`N`, `R`, `Y`).

In [4]:
from be_scan.sgrna._genomic_ import cas_key
for name, pam in cas_key.items():
    print(f"{name:24} {pam}")

Sp                       NGG
SpG                      NGN
SpRY                     NNN
SpRY_highefficiency      NRN
SpRY_lowefficiency       NYN


In [5]:
counts = {}
for cas in ["Sp", "SpG", "SpRY"]:
    g, _ = generate_library(gene_filepath=GENE, gene_name="FOXA1", cas_type=cas,
                            edit_from="A", edit_to="G",
                            return_df=True, save_df=False)
    counts[cas] = len(g)
pd.Series(counts, name="guides found").to_frame()

Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
337 guides were generated
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
2982 guides were generated


,guides found
Sp,337
SpG,992
SpRY,2982


### Filtering options

All four are off by default, so nothing is silently dropped.

| parameter | effect |
|---|---|
| `exclude_introns` | require the editable base to be exonic |
| `exclude_nonediting` | require an editable base in the window at all |
| `exclude_duplicates` | drop spacers that occur more than once |
| `exclude_sequences` | drop spacers containing a motif, e.g. `TTTT` |

In [6]:
rows = []
for label, kw in [
    ("no filters",         dict(exclude_sequences=[])),
    ("drop TTTT",          dict()),
    ("must edit in window",dict(exclude_nonediting=True)),
    ("must edit an exon",  dict(exclude_nonediting=True, exclude_introns=True)),
]:
    g, _ = generate_library(gene_filepath=GENE, gene_name="FOXA1", cas_type="SpG",
                            edit_from="A", edit_to="G",
                            return_df=True, save_df=False, **kw)
    rows.append({"filter": label, "guides": len(g)})
pd.DataFrame(rows)

Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
606 guides were generated
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
580 guides were generated


,filter,guides
0,no filters,992
1,drop TTTT,992
2,must edit in window,606
3,must edit an exon,580


## Step 2 — `annotate`

Predicts, for every guide, which residues the editor could change. This is
the step that has to get intron/exon junctions right: a codon split across a
boundary is completed from the neighbouring exon before translation.

The output columns are prefixed by the edit, so `A`→`G` gives `AtoG_*` and
several editors can be annotated onto one table.

In [7]:
from be_scan.sgrna import annotate

annotated = annotate(
    guides_file = guides,
    edit_from   = "A", edit_to = "G",
    exons       = gene.exons,
    window      = [4, 8],
    return_df   = True, save_df = False,
)
[c for c in annotated.columns if c.startswith("AtoG")]

Guides annotated for A to G.


['AtoG_win_overlap',
 'AtoG_mutations',
 'AtoG_muttypes',
 'AtoG_muttype',
 'AtoG_pos']

In [8]:
annotated[["sgRNA_seq", "sgRNA_strand", "starting_frame",
           "AtoG_win_overlap", "AtoG_mutations", "AtoG_muttype",
           "AtoG_pos"]].head(12)

,sgRNA_seq,sgRNA_strand,starting_frame,AtoG_win_overlap,AtoG_mutations,AtoG_muttype,AtoG_pos
0,gatggttgtattgggcaggg,sense,0,Intron,None,No Mutation,-1.0
1,atggttgtattgggcagggt,sense,1,Intron,None,No Mutation,-1.0
2,gtattgggcagggtggctcc,sense,1,Intron,None,No Mutation,-1.0
3,tattgggcagggtggctcca,sense,2,Intron,None,No Mutation,-1.0
4,tgggcagggtggctccaggA,sense,2,Intron,None,UTR,-1.0
5,cagggtggctccaggATGTT,sense,0,Intron,None,No Mutation,-1.0
6,agggtggctccaggATGTTA,sense,1,Intron,None,No Mutation,-1.0
7,ggctccaggATGTTAGGAAC,sense,0,Intron,None,UTR,-1.0
8,ctccaggATGTTAGGAACTG,sense,2,Exon/Intron,M1V,Missense,1.0
9,caggATGTTAGGAACTGTGA,sense,2,Exon/Intron,M1V,Missense,1.0


Reading the `AtoG_mutations` column: `;` separates alternative edit outcomes,
`/` separates residues changed together by one outcome. A guide with two
editable A's produces three non-identity combinations.

In [9]:
multi = annotated[annotated["AtoG_mutations"].fillna("").str.contains(";")]
multi[["sgRNA_seq", "A_count", "AtoG_mutations", "AtoG_muttype"]].head(6)

,sgRNA_seq,A_count,AtoG_mutations,AtoG_muttype
12,TTAGGAACTGTGAAGATGGA,2,T4A;G3G;G3G/T4A,Missense
13,TAGGAACTGTGAAGATGGAA,2,T4A;G3G;G3G/T4A,Missense
14,AGGAACTGTGAAGATGGAAG,2,T4A;G3G;G3G/T4A,Missense
15,ACTGTGAAGATGGAAGGGCA,2,K6G;K6R;K6E,Missense
16,AGATGGAAGGGCATGAAACC,2,E8G;E8E,Missense
17,ATGGAAGGGCATGAAACCAG,2,E8G;E8E,Missense


### Categories

`*_muttype` collapses the possibilities to one label, in priority order:
splice site, then nonsense, missense, silent, then the non-coding categories.

In [10]:
annotated["AtoG_muttype"].value_counts().to_frame("guides")

,guides
Missense,523
No Mutation,386
Silent,57
UTR,13
Intron,10
Splice-donor,2
Splice-acceptor,1


In [11]:
# guides whose window straddles a junction: the case the annotation has to
# reassemble from two exons
junction = annotated[annotated["AtoG_win_overlap"] == "Exon/Intron"]
print(len(junction), "junction-spanning guides")
junction[["sgRNA_seq", "exon", "AtoG_mutations", "AtoG_muttype"]].head(8)

15 junction-spanning guides


,sgRNA_seq,exon,AtoG_mutations,AtoG_muttype
8,ctccaggATGTTAGGAACTG,0,M1V,Missense
9,caggATGTTAGGAACTGTGA,0,M1V,Missense
34,GCAGGAGgtgagaggcaagg,0,E24G,Missense
35,AGGAGgtgagaggcaaggcg,0,E24G,Missense
36,GGAGgtgagaggcaaggcga,0,,Intron
45,ccagGCCTACTCCTCCGTCC,1,,No Mutation
448,TTCCTAGctcccgggactgg,1,.473W,Missense
449,TCCTAGctcccgggactggg,1,.473W,Missense


### Several editors on one table

Annotating repeatedly adds a new prefixed block each time. `AC`→`GT` is a
dual editor: both substitutions are enumerated together.

In [12]:
for ef, et in [("C", "T"), ("AC", "GT")]:
    annotated = annotate(guides_file=annotated, edit_from=ef, edit_to=et,
                         exons=gene.exons, return_df=True, save_df=False)

pd.DataFrame({
    "ABE (A>G)":  annotated["AtoG_muttype"].value_counts(),
    "CBE (C>T)":  annotated["CtoT_muttype"].value_counts(),
    "dual (AC>GT)": annotated["ACtoGT_muttype"].value_counts(),
}).fillna(0).astype(int)

Guides annotated for C to T.


Guides annotated for AC to GT.


,ABE (A>G),CBE (C>T),dual (AC>GT)
Intron,10,9,1
Missense,523,551,781
No Mutation,386,141,62
Nonsense,0,35,35
Silent,57,233,101
Splice-acceptor,1,3,4
Splice-donor,2,2,2
UTR,13,18,6


### Checking annotations against a protein sequence

Passing `protein_filepath` compares every predicted wild-type residue against
a reference `.fasta` and warns on any disagreement. Worth doing once per gene:
a frame error shows up immediately.

In [13]:
# translate the CDS ourselves and confirm a few calls independently
import re
from be_scan.sgrna._genomic_ import DNA_AA_map

protein = "".join(DNA_AA_map[cds[i:i+3]] for i in range(0, len(cds), 3))
print("protein:", protein[:50], "...")

checked = 0
for cell in annotated["AtoG_mutations"].dropna():
    for mut in re.split(r"[;/]", cell):
        m = re.fullmatch(r"([A-Z.])(\d+)([A-Z.])", mut)
        if m and protein[int(m.group(2))-1] != m.group(1):
            raise AssertionError(f"{mut} disagrees with the translation")
        checked += m is not None
print(f"{checked} mutation calls agree with the translated CDS")

protein: MLGTVKMEGHETSDWNSYYADTQEAYSSVPVSNMNSGLGSMNSMNTYMTM ...
1208 mutation calls agree with the translated CDS


## Step 3 — `reference_check` (optional)

Counts how often each spacer occurs in a reference genome, and can drop
guides that occur more than once. It needs a genome `.fasta`, so it is
skipped here.

```python
from be_scan.sgrna import reference_check

checked = reference_check(
    guides_file = annotated,
    genome_file = "hg38.fa",
    delete      = True,      # drop guides with off-target occurrences
)
```

## Step 4 — `coverage_plots` (optional)

Summarises how much of the protein the library reaches. Needs the protein
`.fasta`.

## All four at once — `design_library`

`design_library` chains generate, annotate, coverage and reference check. Any
optional input left empty skips that step.

In [14]:
from be_scan.sgrna import design_library

library = design_library(
    gene_filepath  = GENE,
    gene_name      = "FOXA1",
    cas_type       = "SpG",
    edit_from_list = ["A", "C", "AC"],
    edit_to_list   = ["G", "T", "GT"],
    window         = [4, 8],
    output_dir     = "output",
    return_df      = True, save_df = False,
)
print(library.shape)
library.head()

Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated


Guides annotated for A to G.


Guides annotated for C to T.

Guides annotated for AC to GT.
Complete! Library generated from data/240401_FOXA1_Input_30.fasta
(992, 32)


,sgRNA_ID,sgRNA_seq,PAM_seq,starting_frame,gene_pos,chr_pos,exon,windowstart_pos,windowend_pos,UTR,coding_seq,sgRNA_strand,gene_strand,gene,AtoG_win_overlap,A_count,AtoG_mutations,AtoG_muttypes,AtoG_muttype,AtoG_pos,CtoT_win_overlap,C_count,CtoT_mutations,CtoT_muttypes,CtoT_muttype,CtoT_pos,ACtoGT_win_overlap,AC_count,ACtoGT_mutations,ACtoGT_muttypes,ACtoGT_muttype,ACtoGT_pos
0,sgRNA_0,gatggttgtattgggcaggg,tgg,0,-1,37594871,0,-1,-1,True,gatggttgtattgggcaggg,sense,minus,FOXA1,Intron,0,NaN,NaN,No Mutation,-1.0,Intron,0,NaN,NaN,No Mutation,-1.0,Intron,0,None,None,No Mutation,-1.0
1,sgRNA_1,atggttgtattgggcagggt,ggc,1,-1,37594870,0,-1,-1,True,atggttgtattgggcagggt,sense,minus,FOXA1,Intron,0,NaN,NaN,No Mutation,-1.0,Intron,0,NaN,NaN,No Mutation,-1.0,Intron,0,None,None,No Mutation,-1.0
2,sgRNA_2,gtattgggcagggtggctcc,agg,1,-1,37594864,0,-1,-1,True,gtattgggcagggtggctcc,sense,minus,FOXA1,Intron,0,NaN,NaN,No Mutation,-1.0,Intron,0,NaN,NaN,No Mutation,-1.0,Intron,0,None,None,No Mutation,-1.0
3,sgRNA_3,tattgggcagggtggctcca,ggA,2,-1,37594863,0,-1,-1,True,tattgggcagggtggctcca,sense,minus,FOXA1,Intron,0,NaN,NaN,No Mutation,-1.0,Intron,0,NaN,NaN,UTR,-1.0,Intron,0,None,None,No Mutation,-1.0
4,sgRNA_4,tgggcagggtggctccaggA,TGT,2,-1,37594860,0,-1,-1,True,tgggcagggtggctccaggA,sense,minus,FOXA1,Intron,0,NaN,NaN,UTR,-1.0,Intron,0,NaN,NaN,UTR,-1.0,Intron,0,None,None,No Mutation,-1.0


## A note on `window`

Before 3.0.0 `window` was documented as tunable but only correct at its
default. It now works for any window with `start >= 3` and `end <= 18`.

Outside that range a codon overlapping guide positions 1-2 or 19-20 can
extend past the end of the guide, and cannot be resolved from the guide
sequence alone; those bases are skipped rather than mis-numbered.

In [15]:
rows = []
for w in [[3, 7], [4, 8], [5, 9], [6, 12], [10, 14]]:
    g, gn = generate_library(gene_filepath=GENE, gene_name="FOXA1", cas_type="SpG",
                             edit_from="A", edit_to="G", window=w,
                             return_df=True, save_df=False)
    a = annotate(guides_file=g, edit_from="A", edit_to="G", exons=gn.exons,
                 window=w, return_df=True, save_df=False)
    rows.append({"window": str(w), "guides": len(g),
                 "missense": (a["AtoG_muttype"] == "Missense").sum(),
                 "nonsense": (a["AtoG_muttype"] == "Nonsense").sum()})
pd.DataFrame(rows)

Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated


Guides annotated for A to G.
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated


Guides annotated for A to G.
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated


Guides annotated for A to G.
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated


Guides annotated for A to G.
Create gene object from data/240401_FOXA1_Input_30.fasta
Parsing exons: 2 exons found
Preprocessing sucessful!
Guides generated and duplicates removed
992 guides were generated


Guides annotated for A to G.


,window,guides,missense,nonsense
0,"[3, 7]",992,518,0
1,"[4, 8]",992,523,0
2,"[5, 9]",992,502,0
3,"[6, 12]",992,621,0
4,"[10, 14]",992,514,0
